# 풀스택 GPT: #6.0~6.10

## Tasks:

- [x] Stuff Documents 체인을 사용하여 완전한 RAG 파이프라인을 구현하세요.
- [x] 체인을 수동으로 구현해야 합니다.
- [x] 체인에 ConversationBufferMemory를 부여합니다.
- [x] [이 문서](https://gist.github.com/serranoarevalo/5acf755c2b8d83f1707ef266b82ea223)를 사용하여 RAG를 수행하세요.
- 체인에 다음 질문을 합니다:
    > - [x] Is Aaronson guilty?
    > - [x] What message did he write in the table?
    > - [x] Who is Julia?
- 다음과 같은 절차대로 구현하면 챌린지를 해결할 수 있습니다.
    > - [x] (1) [문서 로드하기](https://python.langchain.com/v0.1/docs/modules/data_connection/document_loaders/) : TextLoader 등 을 사용해서 파일에서 텍스트를 읽어옵니다.
    > - [x] (2) 문서 쪼개기 : [CharacterTextSplitter](https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/character_text_splitter/) 등 을 사용해서 문서를 작은 문서 조각들로 나눕니다.
    > - [x] (3) 임베딩 생성 및 [캐시](https://python.langchain.com/v0.1/docs/modules/data_connection/text_embedding/caching_embeddings/) : OpenAIEmbeddings, CacheBackedEmbeddings 등 을 사용해 문서 조각들을 임베딩하고 임베딩을 저장합니다.
    > - [x] (4) 벡터 스토어 생성 : [FAISS](https://python.langchain.com/v0.1/docs/integrations/vectorstores/faiss/) 등 을 사용해서 임베딩된 문서들을 저장하고 검색할 수 있는 데이터베이스를 만듭니다.
    > - [x] (5) 대화 메모리와 질문 처리 : ConversationBufferMemory를 사용해 대화 기록을 관리합니다.
    > - [x] (6) 체인 연결 : 앞에서 구현한 컴포넌트들을 적절하게 체인으로 연결합니다.

In [ ]:
import os
from dotenv import load_dotenv
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.vectorstores.faiss import FAISS
from langchain.memory import ConversationBufferMemory
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

load_dotenv()
set_llm_cache(InMemoryCache())
FILE_PATH = "../lectures/#6_files/document.txt"
CACHE_DIR = LocalFileStore("./assignment_4_cache/")

loader = TextLoader(FILE_PATH)
splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=600,
    chunk_overlap=100,
)
file = loader.load_and_split(splitter)

embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, CACHE_DIR)
vectorstore = FAISS.from_documents(file, cached_embeddings)
retriever = vectorstore.as_retriever()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME"),
    temperature=0.1,
)
memory = ConversationBufferMemory(return_messages=True)
prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant. Answer the question only with given context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human","{question}"),
])

def load_memory(_):
    return memory.load_memory_variables({})["history"]

chain = {"context":retriever, "history":RunnableLambda(load_memory), "question":RunnablePassthrough()} | prompt | llm

def invoke_chain(query):
    result = chain.invoke(query)
    memory.save_context({"input":query}, {"output":result.content})

Created a chunk of size 717, which is longer than the specified 600
Created a chunk of size 608, which is longer than the specified 600
Created a chunk of size 642, which is longer than the specified 600
Created a chunk of size 1444, which is longer than the specified 600
Created a chunk of size 1251, which is longer than the specified 600
Created a chunk of size 1012, which is longer than the specified 600
Created a chunk of size 2313, which is longer than the specified 600
Created a chunk of size 1458, which is longer than the specified 600
Created a chunk of size 1673, which is longer than the specified 600
Created a chunk of size 742, which is longer than the specified 600
Created a chunk of size 669, which is longer than the specified 600
Created a chunk of size 906, which is longer than the specified 600
Created a chunk of size 703, which is longer than the specified 600
Created a chunk of size 1137, which is longer than the specified 600
Created a chunk of size 1559, which is lo

In [7]:
invoke_chain("Is Aaronson guilty?")

No, Aaronson is not guilty; the narrator had never seen the photograph that disproved their guilt, and it had never existed.

In [8]:
invoke_chain("What message did he write in the table?")

He wrote "FREEDOM IS SLAVERY" and "TWO AND TWO MAKE FIVE" on the table.

In [9]:
invoke_chain("Who is Julia?")

Julia is a character who is loved by the narrator. She is someone with whom he shares a deep emotional connection, and he expresses a strong desire to protect her.